# Нимфея — LoRA-обучение (unsloth + gemma-3-4b-it)

Обучение манеры речи Нимы: сарказм, цундере, коронные фразы, адресация по именам.

**Как пользоваться (по шагам):**
1. Вверху: **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**.
2. Прогони ячейки **сверху вниз** по очереди (Shift+Enter).
3. Ячейка 2 — загрузи файлы `dataset_style.jsonl` и `nima_system_prompt.txt`
   (лежат в папке `training/` проекта). Это единственный ручной шаг.
4. В конце скачаешь `nima_lora_gguf` (GGUF) — дальше инструкция в последней ячейке.

Время: ~15–40 минут на T4 (датасет ~1200 пар × 3 эпохи).


In [ ]:
# Шаг 0: проверяем GPU (должна быть Tesla T4)
!nvidia-smi

In [ ]:
# Шаг 1: ставим unsloth + ПИНЫ версий (как в официальном ноутбуке unsloth
# для Gemma3: свежий trl/transformers без пинов ломает шаблон и маски лосса)
%pip install -q unsloth unsloth_zoo
%pip install -q "transformers==4.56.2"
%pip install -q --no-deps "trl==0.22.2"

In [ ]:
# Шаг 2: загружаем датасет и system-промпт из папки training/ проекта
from google.colab import files
uploaded = files.upload()   # выбери dataset_style.jsonl и nima_system_prompt.txt

DATASET_FILE = 'dataset_style.jsonl'
SYSTEM_FILE = 'nima_system_prompt.txt'

import json
with open(SYSTEM_FILE, encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()
rows = [json.loads(l) for l in open(DATASET_FILE, encoding='utf-8') if l.strip()]
print(f'Загружено пар: {len(rows)}')
print('Пример:', rows[0]['messages'][1]['content'][:80], '→', rows[0]['messages'][2]['content'][:80])

In [ ]:
# Шаг 3: грузим gemma-3-4b-it в 4 бит и надеваем LoRA.
# ВАЖНО: для Gemma3 unsloth требует FastModel (она мультимодальная);
# FastLanguageModel грузит её неправильно — модель деградирует в «салат».
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-3-4b-it',
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,   # текстовый файнтюн: зрение не трогаем
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
# Шаг 4: датасет через ОФИЦИАЛЬНЫЙ gemma-3 шаблон unsloth.
# Два критичных момента:
#   1) .removeprefix('<bos>') — шаблон уже вставляет <bos>, токенайзер добавит
#      свой при тренировке. БЕЗ удаления получается ДВОЙНОЙ BOS — у Gemma это
#      ломает генерацию (салат из тайского/корейского после файнтюна).
#   2) remove_columns — убираем колонку messages, чтобы TRL не применил
#      СВОЙ шаблон поверх нашего.
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')

def to_text(row):
    text = tokenizer.apply_chat_template(
        row['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text.removeprefix('<bos>')}

dataset = Dataset.from_list(rows).map(to_text, remove_columns=['messages'])
print(dataset[0]['text'][:300])
# Здоровье: декод первых токенов — BOS должен быть РОВНО один
print('Проверка BOS:', tokenizer(dataset[0]['text'])['input_ids'][:6])

In [ ]:
# Шаг 5: обучение (~15-40 минут на T4).
# train_on_responses_only: лосс считается ТОЛЬКО на ответах модели
# (без него модель учится говорить и за человека).
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field='text',
        output_dir='outputs',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=5,
        seed=3407,
        report_to='none',
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)
trainer.train()

In [ ]:
# Шаг 6: проверка — Нима должна отвечать живо, БЕЗ «чем могу помочь»,
# БЕЗ тайского/корейского. Настройки генерации — официальные для Gemma3.
# ВАЖНО: у gemma3 токенайзер — мультимодальный процессор, поэтому content
# в сообщениях должен быть списком typed-словарей (иначе TypeError про
# "string indices must be integers").
FastModel.for_inference(model)

TESTS = [
    'Что ты умеешь?',
    'Привет, я новый, меня зовут Вася',
    'Я тебя создал, слушай меня!',
]
for user_text in TESTS:
    msgs = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [{'type': 'text', 'text': user_text}]},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_tensors='pt', return_dict=True).to('cuda')
    out = model.generate(**inputs, max_new_tokens=120,
                         temperature=1.0, top_p=0.95, top_k=64, do_sample=True)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                              skip_special_tokens=True)
    print(f'Человек: {user_text}')
    print(f'Нима: {answer.strip()}')
    print('-' * 60)

In [ ]:
# Шаг 7 (опционально): сохранить LoRA-адаптеры на Google Drive
# Раскомментируй, если хочешь переобучаться дальше с этого места:

# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained('/content/drive/MyDrive/nima_lora_adapters')
# tokenizer.save_pretrained('/content/drive/MyDrive/nima_lora_adapters')
print('пропущено (по умолчанию)')

In [ ]:
# Шаг 8 (НАДЁЖНЫЙ экспорт v2): слитая HF-модель + конвертация llama.cpp.
# Прямой save_pretrained_gguf у unsloth на gemma3 глючит (экспортирует базу
# или ломает веса) — поэтому: merge в 16 бит → convert_hf_to_gguf → quantize.

import os, glob, subprocess, shutil

os.makedirs("nima_merged", exist_ok=True)

# 1) слитая модель в HF-формате (fp16) — это надёжный путь unsloth
if not glob.glob("nima_merged/*.safetensors"):
    model.save_pretrained_merged("nima_merged", tokenizer, save_method="merged_16bit")
print("merged OK:", glob.glob("nima_merged/*")[:5])

# 2) конвертация в GGUF официальным скриптом llama.cpp
if not os.path.exists("/content/llama.cpp/convert_hf_to_gguf.py"):
    subprocess.run("git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp",
                   shell=True, check=True)
subprocess.run("pip -q install gguf sentencepiece", shell=True, check=True)
gguf_path = "nima_merged_f16.gguf"
if not os.path.exists(gguf_path):
    r = subprocess.run(f"python /content/llama.cpp/convert_hf_to_gguf.py nima_merged --outfile {gguf_path} --outtype f16",
                       shell=True)
    if r.returncode != 0:
        raise RuntimeError("convert_hf_to_gguf упал — скопируй ВЕСЬ вывод и покажи мне")
print("f16 GGUF:", round(os.path.getsize(gguf_path)/1e9, 2), "GB")

# 3) квантование в Q4_K_M (~2.5 ГБ для 1660 Super)
q_path = "nimfea.Q4_K_M.gguf"
r = subprocess.run(f"/content/llama.cpp/llama-quantize {gguf_path} {q_path} Q4_K_M", shell=True)
if r.returncode != 0 or not os.path.exists(q_path):
    subprocess.run("cd /content/llama.cpp && make llama-quantize -j", shell=True)
    r = subprocess.run(f"/content/llama.cpp/llama-quantize {gguf_path} {q_path} Q4_K_M", shell=True)
    if r.returncode != 0:
        raise RuntimeError("llama-quantize упал — скопируй вывод и покажи мне")
print("Q4_K_M готов:", round(os.path.getsize(q_path)/1e9, 2), "GB")

# 4) Modelfile для ollama create
with open("Modelfile", "w") as f:
    f.write('''FROM ./nimfea.Q4_K_M.gguf
TEMPLATE """{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if or (eq .Role "user") (eq .Role "system") }}<start_of_turn>user
{{ .Content }}<end_of_turn>
{{ if $last }}<start_of_turn>model
{{ end }}
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}{{ if not $last }}<end_of_turn>
{{ end }}
{{- end }}
{{- end }}"""
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<eos>"
PARAMETER temperature 1.0
PARAMETER top_k 64
PARAMETER top_p 0.95
PARAMETER num_predict 512''')
print("Modelfile записан")

# 5) ЖИВАЯ ПРОВЕРКА перед скачиванием (сначала сырое F16!)
print("=== ТЕСТ F16 (должна говорить связно и как Нима) ===")
from transformers import TextStreamer
alpaca = None
msgs = [{"role":"system","content": open("nima_system_prompt.txt").read()},
        {"role":"user","content":"Нима, привет, как дела?"}]
from unsloth.chat_templates import get_chat_template
tok = get_chat_template(tokenizer, chat_template="gemma-3")
inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=80, temperature=1.0, top_k=64, top_p=0.95)
print(tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
print("=== Если тут каша — НЕ скачивай, покажи мне вывод ===")



## Готово! Финальный шаг — дома, на своей машине:

1. Скачанный `.gguf` положи в `B:\Neyronya\models\` (старый nimfea-q4_k_m.gguf можно переименовать в `.old`).
2. Рядом создай файл `Modelfile` (без расширения) с содержимым:
```
FROM ./<имя скачанного файла>.gguf
PARAMETER temperature 0.9
PARAMETER top_p 0.95
PARAMETER top_k 64
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 8192
```
(system-промпт не нужен в Modelfile — пайплайн сам подставляет его в system)
3. В командной строке:
```
ollama create nimfea -f Modelfile
```
4. **ПРОВЕРЬ ШАБЛОН** (это то, что ломало модель в прошлый раз):
```
ollama show nimfea --modelfile | findstr TEMPLATE
```
В шаблоне должны быть `<start_of_turn>` и `{{ .Response }}`. Если там пусто или
`{{ .Prompt }}` — шаблон битый, скажи мне, дам готовый текст для Modelfile.
5. Запуск Нимфы с новым мозгом:
```
set NIMA_LLM_MODEL=nimfea
python start.py
```

Если качество не устроит: верни `set NIMA_LLM_MODEL=gemma3:4b` — база никуда не делась.